# Reproducible Analytical Workflow

This notebook runs the reusable analytical backend on the existing processed v2.0 transactions. It produces customer features and behavioral clusters, transaction anomaly predictions, statistical summaries, label-based anomaly metrics, and traceable rule-based insights. The functions live in `src/`; the notebook only loads data, calls them, and reviews their outputs.

The synthetic anomaly labels are kept separate from model inputs. The raw and processed data files are read only.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repository_root = Path.cwd()
if not (repository_root / "src").exists():
    repository_root = repository_root.parent

sys.path.insert(0, str(repository_root))

from src.analytics_pipeline import run_analytical_pipeline

transactions = pd.read_csv(repository_root / "data/processed/enriched_transactions.csv")
anomaly_labels = pd.read_csv(repository_root / "data/reference/synthetic_anomaly_labels.csv")
customer_reference = pd.read_csv(repository_root / "data/reference/synthetic_customers.csv")

print("Transaction shape:", transactions.shape)
print("Customer count:", transactions["customer_id"].nunique())
print("Date range:", transactions["date"].min(), "to", transactions["date"].max())
print("Injected anomaly labels:", int(anomaly_labels["is_injected_anomaly"].sum()))

Transaction shape: (56421, 14)
Customer count: 180
Date range: 2025-01-01 to 2025-12-31
Injected anomaly labels: 354


## Run all analytical components

The pipeline returns separate customer-level and transaction-level tables. `customer_id` is the customer grain for features and clusters. `transaction_id` is the transaction grain for anomaly predictions. The optional label table is passed only to evaluation after prediction.

In [1]:
analysis = run_analytical_pipeline(
    transactions,
    anomaly_labels=anomaly_labels,
    customer_reference=customer_reference,
    contamination=0.01,
    random_state=42,
)

print("Selected behavioral cluster count:", analysis["selected_cluster_count"])
print("Customer feature rows:", len(analysis["customer_features"]))
print("Customer segment rows:", len(analysis["customer_segments"]))
print("Transaction prediction rows:", len(analysis["anomaly_predictions"]))
print("Anomaly model feature columns:", analysis["anomaly_features"].columns.tolist())
print("Label column used as model input:", "is_injected_anomaly" in analysis["anomaly_features"].columns)

Selected behavioral cluster count: 2
Customer feature rows: 180
Customer segment rows: 180
Transaction prediction rows: 56421
Anomaly model feature columns: ['transaction_id', 'customer_id', 'date', 'amount', 'category', 'transaction_type', 'amount_magnitude', 'customer_prior_median', 'category_prior_median', 'reference_amount', 'customer_amount_ratio', 'category_amount_ratio', 'historical_amount_ratio', 'log_amount_magnitude', 'log_customer_amount_ratio', 'log_category_amount_ratio', 'baseline_source', 'is_eligible']
Label column used as model input: False


## Customer segmentation

The selected cluster count is based on the largest silhouette score among candidates from 2 through 10. Review the candidate scores, group sizes, and profiles together; a score alone does not establish business usefulness. Profiles report means and medians in original units.

In [1]:
display(analysis["cluster_evaluation"].round(4))

display(analysis["customer_segments"]["behavioral_cluster"].value_counts().sort_index())
display(analysis["cluster_profiles"].round(2))

,k,inertia,silhouette_score
0,2,1627.6839,0.4208
1,3,1393.4962,0.2133
2,4,1221.0360,0.2168
3,5,1154.0308,0.2064
4,6,1102.9169,0.1471
5,7,1049.9000,0.1420
6,8,1008.0198,0.1353
7,9,966.2902,0.1432
8,10,928.4603,0.1378


behavioral_cluster
0    140
1     40
Name: count, dtype: int64

share_Bills & Utilities  ... weekend_transaction_share
                                      mean  ...                    median
behavioral_cluster                          ...                          
0                                     4.62  ...                      0.28
1                                     5.42  ...                      0.27

[2 rows x 30 columns]

## Transaction anomaly evaluation

The Isolation Forest models use a chronological 80% training period and score later transactions. The contamination setting is 1%. The evaluation below reports precision, recall, F1, and average precision because the injected anomaly labels are rare. The confusion matrix and precision-recall table support error review.

In [1]:
print("Training cutoff:", analysis["anomaly_training_cutoff"])
display(analysis["anomaly_evaluation"]["metrics"].round(4))
display(analysis["anomaly_evaluation"]["confusion_matrix"])
display(analysis["anomaly_evaluation"]["precision_recall_curve"].head())

Training cutoff: 2025-10-19 00:00:00


,evaluated_transactions,actual_anomalies,predicted_anomalies,precision,recall,f1_score,average_precision
0,11114,63,117,0.3162,0.5873,0.4111,0.4444


,predicted_normal,predicted_anomaly
actual_normal,10971,80
actual_anomaly,26,37


,precision,recall,threshold
0,0.005669,1.0,-0.287652
1,0.005669,1.0,-0.286890
2,0.005670,1.0,-0.286174
3,0.005670,1.0,-0.286110
4,0.005671,1.0,-0.285810


## Statistical summaries and group comparison

The amount distribution and IQR bounds are descriptive checks, not transaction fraud labels. The group comparison uses one average debit amount per customer, so repeated transactions are not treated as independent observations.

Research question: do customers in the synthetic student and salaried-individual profiles have different average debit transaction amounts? The null hypothesis is equal group means; the alternative is different means. Welch's two-sample t-test allows unequal variances. The reported Shapiro-Wilk p-values are a small-sample check on each customer-level distribution. Results describe these synthetic customers only.

In [1]:
display(analysis["statistical_findings"]["debit_amount_descriptive_statistics"].round(2))
print("IQR outlier bounds:")
print(analysis["statistical_findings"]["debit_amount_iqr_bounds"])

group_comparison = analysis["statistical_findings"]["customer_group_comparison"]
display(group_comparison.T)

IQR outlier bounds:
{'first_quartile': np.float64(515.3425), 'third_quartile': np.float64(2084.6625), 'interquartile_range': np.float64(1569.32), 'lower_bound': np.float64(-1838.6375), 'upper_bound': np.float64(4438.6425), 'outlier_count': 5062}


,count,mean,median,standard_deviation,minimum,first_quartile,third_quartile,maximum,skewness,interquartile_range
0,53274,2278.96,1025.24,4685.97,22.85,515.34,2084.66,139633.29,6.76,1569.32


,0
unit_of_analysis,customer
measure,average_debit_amount
first_group,student
first_group_count,40
first_group_mean,1174.971212
first_group_median,1189.316838
first_group_skewness,-0.18532
first_group_shapiro_pvalue,0.785859
second_group,salaried_individual
second_group_count,47


## Traceable insights

Insights are generated from calculated category shares, cluster transaction counts, and model predictions. Each row keeps the customer ID, transaction ID, category, cluster, metric, and measured value where they apply.

In [1]:
print("Insight count:", len(analysis["insights"]))
display(analysis["insights"].head(12))

display(analysis["insights"]["insight_type"].value_counts())

Insight count: 686


,insight_type,customer_id,transaction_id,cluster,category,metric,value,comparison_value,source_explanation,insight_text
0,concentrated_category_share,CUST0063,NaN,NaN,Housing,category_spending_share_percent,44.502615,NaN,NaN,Housing was 44.5% of this customer's recorded ...
1,concentrated_category_share,CUST0134,NaN,NaN,Housing,category_spending_share_percent,47.655642,NaN,NaN,Housing was 47.7% of this customer's recorded ...
2,concentrated_category_share,CUST0104,NaN,NaN,Housing,category_spending_share_percent,31.058811,NaN,NaN,Housing was 31.1% of this customer's recorded ...
3,concentrated_category_share,CUST0176,NaN,NaN,Housing,category_spending_share_percent,45.877540,NaN,NaN,Housing was 45.9% of this customer's recorded ...
4,concentrated_category_share,CUST0166,NaN,NaN,Housing,category_spending_share_percent,32.142566,NaN,NaN,Housing was 32.1% of this customer's recorded ...
5,concentrated_category_share,CUST0078,NaN,NaN,Housing,category_spending_share_percent,30.657798,NaN,NaN,Housing was 30.7% of this customer's recorded ...
6,concentrated_category_share,CUST0069,NaN,NaN,Housing,category_spending_share_percent,41.540395,NaN,NaN,Housing was 41.5% of this customer's recorded ...
7,concentrated_category_share,CUST0044,NaN,NaN,Housing,category_spending_share_percent,46.330456,NaN,NaN,Housing was 46.3% of this customer's recorded ...
8,concentrated_category_share,CUST0009,NaN,NaN,Housing,category_spending_share_percent,38.204206,NaN,NaN,Housing was 38.2% of this customer's recorded ...
9,concentrated_category_share,CUST0095,NaN,NaN,Housing,category_spending_share_percent,41.392401,NaN,NaN,Housing was 41.4% of this customer's recorded ...


insight_type
transaction_anomaly              556
concentrated_category_share      128
cluster_transaction_frequency      2
Name: count, dtype: int64

## Reproduction and limits

Run from the repository root with:

```text
.\venv\Scripts\python.exe -m pytest tests -q
```

The pipeline accepts the already processed transaction DataFrame and returns analytical outputs without writing files. Its anomaly baselines require historical observations and fall back from customer-category history to category-direction history. The 1% contamination assumption affects how many records are flagged. Synthetic labels and profiles do not represent independently sampled real customers. The clustering and statistical comparisons are exploratory. An anomaly is unusual relative to the selected reference features; it is not a fraud determination. Streamlit is not implemented.